<a id="setup"></a>
# SentencePiece — Option A : Broadcasting du tag

Chaque sous-token d'un mot reçoit le même tag que le mot entier. C'est l'approche la plus simple : elle demande de changer la façon dont les tokens sont produits, mais rien dans l'architecture du modèle ni dans la boucle d'entraînement.

In [8]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent.parent))
from bambara_pos_utils import *

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, accuracy_score
import sentencepiece as spm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device :", device)

SP_DIR = Path.cwd().parent  # 07_sentencepiece_tokenization/

Device : cpu


In [9]:
# Chargement du corpus et split identique aux modules 05/06 (même seed)
corpus_path = Path.cwd().parent.parent / "04_bambara_pos_pipeline" / "bambara_pos_prep_retagged.conll"
corpus_data = load_bambara_corpus(corpus_path)
train_data, val_data, test_data = split_corpus(corpus_data, seed=42)
print(f"Train : {len(train_data)} | Val : {len(val_data)} | Test : {len(test_data)}")

Corpus chargé : 37392 phrases.
Train : 29913 | Val : 3739 | Test : 3740


<a id="train-sp"></a>
# Entraînement du modèle SentencePiece

Entraîné uniquement sur le texte brut (les mots, sans les tags). vocab_size=4000 est un point de départ raisonnable pour un corpus de cette taille (640k tokens) — un vocabulaire trop grand redeviendrait proche du mot-entier, un vocabulaire trop petit fragmenterait excessivement les mots.

In [10]:
sp_model_prefix = SP_DIR / "bambara_sp"

if not (SP_DIR / "bambara_sp.model").is_file():
    raw_text_path = SP_DIR / "bambara_raw_text.txt"
    export_raw_text(corpus_data, raw_text_path)

    spm.SentencePieceTrainer.train(
        input=str(raw_text_path),
        model_prefix=str(sp_model_prefix),
        vocab_size=4000,
        model_type="unigram",
        pad_id=0, unk_id=1,
        bos_id=-1, eos_id=-1,
    )
    print("Modèle SentencePiece entraîné.")
else:
    print("Modèle SentencePiece déjà présent, réutilisation.")

sp = spm.SentencePieceProcessor(model_file=str(SP_DIR / "bambara_sp.model"))
print("Taille du vocabulaire de sous-mots :", sp.get_piece_size())

Modèle SentencePiece déjà présent, réutilisation.
Taille du vocabulaire de sous-mots : 4000


In [11]:
# Vérification rapide de la tokenisation en sous-mots
exemple = sp.encode("bamanankan", out_type=str)
print("bamanankan ->", exemple)

exemple2 = sp.encode("Amadou bɛ kalan kɛ", out_type=str)
print("phrase complète ->", exemple2)

bamanankan -> ['▁bamanankan']
phrase complète -> ['▁A', 'ma', 'do', 'u', '▁bɛ', '▁kalan', '▁kɛ']


<a id="alignment"></a>
# Alignement sous-tokens / tags (broadcasting)

In [12]:
# Test sur une phrase du corpus, pour vérifier l'alignement avant d'entraîner
words, tags = train_data[0]
subword_tokens, subword_tags = encode_sentence_with_tags(words, tags, sp, scheme="broadcast")

print("Mots originaux :", words)
print("Tags originaux :", tags)
print()
for tok, tag in zip(subword_tokens, subword_tags):
    print(f"  {tok:<12} {tag}")

Mots originaux : ['Ani', 'ne', 'ye', 'buurukun', 'wolonwula', 'kari', 'kari', 'ka', 'di', 'mɔgɔ', 'ba', 'naani', 'ma', 'tuma', 'min', ',', 'aw', 'ye', 'segi', 'joli', 'fa', 'kunkuruw', 'tɔ', 'la', '?']
Tags originaux : ['CONJ', 'PRON', 'AUX', 'VERBE', 'NOM', 'NOM', 'NOM', 'AUX', 'VERBE', 'NOM', 'NOM', 'NOM', 'AUX', 'NOM', 'PRON', 'PUNCT', 'PRON', 'AUX', 'VERBE', 'NOM', 'NOM', 'NOM', 'NOM', 'POSTP', 'PUNCT']

  ▁Ani         CONJ
  ▁ne          PRON
  ▁ye          AUX
  ▁buurukun    VERBE
  ▁wolonwula   NOM
  ▁kari        NOM
  ▁kari        NOM
  ▁ka          AUX
  ▁di          VERBE
  ▁mɔgɔ        NOM
  ▁ba          NOM
  ▁naani       NOM
  ▁ma          AUX
  ▁tuma        NOM
  ▁min         PRON
  ▁,           PUNCT
  ▁aw          PRON
  ▁ye          AUX
  ▁segi        VERBE
  ▁joli        NOM
  ▁fa          NOM
  ▁kun         NOM
  kuru         NOM
  w            NOM
  ▁tɔ          NOM
  ▁la          POSTP
  ▁?           PUNCT


<a id="training"></a>
# Entraînement du modèle BiLSTM sur les sous-tokens

Avec le broadcasting, le tagset reste inchangé (les 12 tags habituels : label2id). Seul le vocabulaire d'entrée change — ce n'est plus word_to_ix mais l'ID de sous-mot SentencePiece.

In [13]:
hyperparams = {
    "learning_rate": 1e-3,
    "batch_size": 32,
    "hidden_dim": 128,
    "embedding_dim": 100,
}

MODELS_DIR = Path.cwd() / "models"
MODELS_DIR.mkdir(exist_ok=True)
checkpoint_path = MODELS_DIR / "bambara_pos_sp_broadcast.pth"

model, history, best_val_loss = run_training(
    train_data, val_data,
    dataset_cls=BambaraSubwordDataset,
    dataset_args=(sp, label2id, "broadcast"),
    vocab_size=sp.get_piece_size(),
    tagset_size=len(label2id),
    hyperparams=hyperparams,
    device=device,
    patience=4, max_epochs=30,
    checkpoint_path=checkpoint_path,
)

Époque  1 | Train Loss : 0.1681 | Val Loss : 0.0353 | Val Acc : 98.89%
Époque  2 | Train Loss : 0.0192 | Val Loss : 0.0164 | Val Acc : 99.53%
Époque  3 | Train Loss : 0.0058 | Val Loss : 0.0076 | Val Acc : 99.78%
Époque  4 | Train Loss : 0.0021 | Val Loss : 0.0062 | Val Acc : 99.85%
Époque  5 | Train Loss : 0.0009 | Val Loss : 0.0055 | Val Acc : 99.87%
Époque  6 | Train Loss : 0.0005 | Val Loss : 0.0045 | Val Acc : 99.88%
Époque  7 | Train Loss : 0.0005 | Val Loss : 0.0047 | Val Acc : 99.89%
Époque  8 | Train Loss : 0.0005 | Val Loss : 0.0039 | Val Acc : 99.90%
Époque  9 | Train Loss : 0.0001 | Val Loss : 0.0041 | Val Acc : 99.91%
Époque 10 | Train Loss : 0.0001 | Val Loss : 0.0034 | Val Acc : 99.92%
Époque 11 | Train Loss : 0.0003 | Val Loss : 0.0040 | Val Acc : 99.92%
Époque 12 | Train Loss : 0.0000 | Val Loss : 0.0035 | Val Acc : 99.93%
Époque 13 | Train Loss : 0.0000 | Val Loss : 0.0037 | Val Acc : 99.93%
Époque 14 | Train Loss : 0.0000 | Val Loss : 0.0037 | Val Acc : 99.93%
Arrêt 

<a id="evaluation"></a>
# Évaluation au niveau mot

Les prédictions du modèle sont au niveau sous-token ; on les regroupe par mot avec reconstruct_word_tags pour comparer équitablement avec les résultats mot-entier des modules 05/06.

In [14]:
def evaluate_subword_model_word_level(model, sentences, sp, id2label, scheme, device):
    model.eval()
    all_true, all_pred = [], []

    with torch.no_grad():
        for words, tags in sentences:
            subword_tokens, _ = encode_sentence_with_tags(words, tags, sp, scheme)
            if not subword_tokens:
                continue
            ids = [sp.piece_to_id(t) for t in subword_tokens]
            input_tensor = torch.tensor([ids], dtype=torch.long).to(device)
            preds = torch.argmax(model(input_tensor), dim=-1).squeeze(0).cpu().tolist()
            predicted_tags = [id2label[p] for p in preds]

            reconstructed = reconstruct_word_tags(subword_tokens, predicted_tags, scheme)
            # sécurité : si la reconstruction ne retombe pas sur le même nombre de mots
            n = min(len(reconstructed), len(tags))
            all_true.extend(tags[:n])
            all_pred.extend(reconstructed[:n])

    accuracy = accuracy_score(all_true, all_pred)
    report = classification_report(all_true, all_pred, zero_division=0)
    return accuracy, report


test_acc, test_report = evaluate_subword_model_word_level(
    model, test_data, sp, id2label, "broadcast", device
)
print(f"Accuracy au niveau mot (SentencePiece, broadcast) : {test_acc*100:.2f}%\n")
print(test_report)

Accuracy au niveau mot (SentencePiece, broadcast) : 99.96%

              precision    recall  f1-score   support

         ADJ       0.99      1.00      1.00       251
         ADV       1.00      1.00      1.00       252
         AUX       1.00      1.00      1.00      9650
        CONJ       1.00      1.00      1.00      2812
         DET       1.00      1.00      1.00      2516
         NOM       1.00      1.00      1.00     18577
        PART       1.00      1.00      1.00       841
       POSTP       1.00      1.00      1.00      3411
        PRON       1.00      1.00      1.00      9114
       PUNCT       1.00      1.00      1.00     10488
       VERBE       1.00      1.00      1.00      6444

    accuracy                           1.00     64356
   macro avg       1.00      1.00      1.00     64356
weighted avg       1.00      1.00      1.00     64356



## **Interprétation**

Comparer cette accuracy à celle du module 06 (mot-entier). Si SentencePiece améliore surtout les phrases contenant des mots rares/longs (vérifiable en filtrant test_data par longueur de mot), c'est un signe que l'approche remplit son objectif — réduire l'impact des mots inconnus. Garder en tête que le corpus reste celui, potentiellement bruité, du module 04.